# Breast Cancer (VinDr-Mammo) — Qwen2.5-VL Models A and B

Pipeline:
1. Paths + load CSVs
2. Grouped labels (top findings + Other)
3. Train/val/test splits (VinDr official + carved val)
4. Box scaling helper (PNG-scaled, normalized 0–1000)
5. Build instruction JSONLs (plain + grounded)
6. YOLO grouped data prep
7. Train Qwen2.5-VL-7B Model A (plain grouped)
8. Train Qwen2.5-VL-7B Model B (grounded grouped)
9. Inference
10. Evaluation

All runs in-notebook on H200.

## 1. Setup & paths

In [ ]:
import os, json, ast, re, random, shutil
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from tqdm.auto import tqdm
import torch

DATA_ROOT = Path(os.environ.get('VINDR_ROOT', 'data/vindr_mammo'))  # see README -> Data
RAW       = DATA_ROOT / 'vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0'
PNG       = DATA_ROOT / 'png'
ANN       = DATA_ROOT / 'annotations_processed'
LORA_DATA = DATA_ROOT / 'lora_data'
LORA_RUNS = DATA_ROOT / 'lora_runs'
YOLO_DIR  = DATA_ROOT / 'yolo'
RUNS      = DATA_ROOT / 'runs'

for d in [ANN, LORA_DATA, LORA_RUNS, YOLO_DIR, RUNS]:
    d.mkdir(parents=True, exist_ok=True)

BREAST_CSV  = RAW / 'breast-level_annotations.csv'
FINDING_CSV = RAW / 'finding_annotations.csv'

breast_df  = pd.read_csv(BREAST_CSV)
finding_df = pd.read_csv(FINDING_CSV)

# drop the one corrupt image (failed DICOM->PNG conversion)
BAD_IDS = {'3e71b089f6e55d5791771b05f88518dd'}
breast_df  = breast_df[~breast_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)
finding_df = finding_df[~finding_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)

print('Breast rows:', len(breast_df), ' Finding rows:', len(finding_df))
print('PNG files:', len(list(PNG.glob('*.png'))))
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

Breast rows: 19999  Finding rows: 20485
PNG files: 19999
CUDA: True | NVIDIA H200 NVL


## 2. Grouped labels

multi-label, with rare findings collapsed into clinically meaningful groups:

| Group | Includes | Why |
|---|---|---|
| Mass | Mass | 1,123+ |
| Suspicious Calcification | Suspicious Calcification | 402+ |
| Asymmetry | Asymmetry, Focal Asymmetry, Global Asymmetry | clinically related, individually small |
| Architectural Distortion | Architectural Distortion | important cancer sign |
| Suspicious Lymph Node | Suspicious Lymph Node | clinically meaningful |
| Other Suspicious Signs | Skin Thickening, Skin Retraction, Nipple Retraction | too rare individually |

In [ ]:
GROUPS = {
    'Mass':                     ['Mass'],
    'Suspicious Calcification': ['Suspicious Calcification'],
    'Asymmetry':                ['Asymmetry', 'Focal Asymmetry', 'Global Asymmetry'],
    'Architectural Distortion': ['Architectural Distortion'],
    'Suspicious Lymph Node':    ['Suspicious Lymph Node'],
    'Other Suspicious Signs':   ['Skin Thickening', 'Skin Retraction', 'Nipple Retraction'],
}

GROUPED_CLASSES = list(GROUPS.keys())

GROUP_TO_PHRASE = {
    'Mass': 'A suspicious mass is present',
    'Suspicious Calcification': 'Suspicious calcifications are present',
    'Asymmetry': 'An asymmetry is present',
    'Architectural Distortion': 'Architectural distortion is present',
    'Suspicious Lymph Node': 'A suspicious lymph node is present',
    'Other Suspicious Signs': 'Other suspicious mammographic signs are present',
}

GROUP_TO_KNOWLEDGE = {
    'Mass': 'A mammographic mass is a space-occupying lesion with a visible shape, margin, and density.',
    'Suspicious Calcification': 'Suspicious calcifications appear as small high-density bright foci, often grouped or clustered.',
    'Asymmetry': 'Asymmetry is an area of denser breast tissue without a clear mass, possibly focal or global.',
    'Architectural Distortion': 'Architectural distortion appears as pulling or disruption of normal breast tissue without a definite mass.',
    'Suspicious Lymph Node': 'A suspicious lymph node appears enlarged, dense, rounded, or with abnormal morphology.',
    'Other Suspicious Signs': 'Other suspicious signs include skin thickening, nipple retraction, or skin retraction.',
}

def parse_finding_categories(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        v = ast.literal_eval(str(x))
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [str(x)]

def map_to_groups(categories):
    cats = parse_finding_categories(categories)
    if 'No Finding' in cats:
        return []
    out = set()
    for g, raws in GROUPS.items():
        for r in raws:
            if r in cats:
                out.add(g)
    return sorted(out)

image_to_groups = defaultdict(set)
for _, r in finding_df.iterrows():
    for g in map_to_groups(r['finding_categories']):
        image_to_groups[r['image_id']].add(g)
image_to_groups = {k: sorted(v) for k, v in image_to_groups.items()}

counts = Counter()
for groups in image_to_groups.values():
    for g in groups:
        counts[g] += 1

n_normal = len(breast_df) - sum(1 for img in breast_df['image_id'] if img in image_to_groups and len(image_to_groups[img]) > 0)
print(f'{"Normal (no finding)":30s} {n_normal}')
for g in GROUPED_CLASSES:
    print(f'{g:30s} {counts[g]}')

Normal (no finding)            18231
Mass                           1113
Suspicious Calcification       442
Asymmetry                      390
Architectural Distortion       119
Suspicious Lymph Node          53
Other Suspicious Signs         86


## 3. Splits

In [ ]:
SEED = 0
VAL_FRAC = 0.10

train_studies_all = sorted(breast_df.loc[breast_df['split'] == 'training', 'study_id'].unique().tolist())
test_studies      = sorted(breast_df.loc[breast_df['split'] == 'test',     'study_id'].unique().tolist())

rng = random.Random(SEED)
rng.shuffle(train_studies_all)
n_val = int(len(train_studies_all) * VAL_FRAC)
val_studies   = sorted(train_studies_all[:n_val])
train_studies = sorted(train_studies_all[n_val:])

def studies_to_images(study_ids):
    return sorted(breast_df.loc[breast_df['study_id'].isin(study_ids), 'image_id'].unique().tolist())

train_ids = studies_to_images(train_studies)
val_ids   = studies_to_images(val_studies)
test_ids  = studies_to_images(test_studies)

print(f'Train studies: {len(train_studies):5d}  images: {len(train_ids):5d}')
print(f'Val   studies: {len(val_studies):5d}  images: {len(val_ids):5d}')
print(f'Test  studies: {len(test_studies):5d}  images: {len(test_ids):5d}')

assert not (set(train_studies) & set(val_studies))
assert not (set(train_studies) & set(test_studies))
assert not (set(val_studies)   & set(test_studies))

(ANN / 'splits.json').write_text(json.dumps({
    'train_studies': train_studies, 'val_studies': val_studies, 'test_studies': test_studies,
    'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids,
}))
print('Saved splits to', ANN / 'splits.json')

Train studies:  3600  images: 14399
Val   studies:   400  images:  1600
Test  studies:  1000  images:  4000
Saved splits to $VINDR_ROOT/annotations_processed/splits.json


## 4. Box scaling


In [ ]:
_png_dim_cache = {}

def get_png_size(image_id):
    if image_id in _png_dim_cache:
        return _png_dim_cache[image_id]
    p = PNG / f'{image_id}.png'
    if not p.exists():
        return None
    with PILImage.open(p) as im:
        _png_dim_cache[image_id] = im.size  # (W, H)
    return _png_dim_cache[image_id]

def box_to_norm1000(row, image_id):
    """Original DICOM coords -> PNG-scaled -> normalized 0-1000.
    Returns [x1, y1, x2, y2] ints, or None if invalid."""
    size = get_png_size(image_id)
    if size is None: return None
    png_w, png_h = size

    orig_w, orig_h = float(row['width']), float(row['height'])
    sx, sy = png_w / orig_w, png_h / orig_h

    x1 = float(row['xmin']) * sx
    y1 = float(row['ymin']) * sy
    x2 = float(row['xmax']) * sx
    y2 = float(row['ymax']) * sy

    x1 = max(0, min(png_w, x1)); x2 = max(0, min(png_w, x2))
    y1 = max(0, min(png_h, y1)); y2 = max(0, min(png_h, y2))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None

    return [
        int(round(x1 / png_w * 1000)),
        int(round(y1 / png_h * 1000)),
        int(round(x2 / png_w * 1000)),
        int(round(y2 / png_h * 1000)),
    ]

image_to_boxes = defaultdict(list)
for _, r in tqdm(finding_df.iterrows(), total=len(finding_df), desc='Scaling boxes'):
    if pd.isna(r.get('xmin', np.nan)): continue
    groups = map_to_groups(r['finding_categories'])
    if not groups: continue
    box = box_to_norm1000(r, r['image_id'])
    if box is None: continue
    for g in groups:
        image_to_boxes[r['image_id']].append({'group': g, 'box': box})

print(f'Images with at least one grouped box: {len(image_to_boxes)}')

sample_id = next(iter(image_to_boxes))
print(f'\nSample {sample_id}: {image_to_boxes[sample_id]}')

Images with at least one grouped box: 1768

Sample <image_id>: [{'group': 'Mass', 'box': [841, 492, 887, 527]}]


## 5. Build instruction JSONLs

In [ ]:
PLAIN_DIR = LORA_DATA / 'plain_grouped'
GND_DIR   = LORA_DATA / 'grounded_grouped'
PLAIN_DIR.mkdir(parents=True, exist_ok=True)
GND_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_MAMMO = (
    'You are an expert radiology assistant for mammography. '
    'Answer carefully using only the image information.'
)

def get_breast_row(image_id):
    rows = breast_df[breast_df['image_id'] == image_id]
    return rows.iloc[0] if len(rows) else None

def make_messages(image_path, user_text, assistant_text):
    return [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_MAMMO}]},
        {'role': 'user',      'content': [{'type': 'image', 'image': str(image_path)},
                                          {'type': 'text', 'text': user_text}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
    ]

def finding_classification_answer(groups):
    if not groups: return 'No suspicious mammographic finding is present.'
    return 'The suspicious mammographic findings present are: ' + ', '.join(groups) + '.'

def birads_density_answer(row):
    return f'The BI-RADS category is {row["breast_birads"]}. The breast density is {row["breast_density"]}.'

def plain_report_answer(groups, row):
    parts = [birads_density_answer(row)]
    if not groups:
        parts.append('No suspicious mammographic finding is present.')
    else:
        for g in groups:
            parts.append(GROUP_TO_PHRASE[g] + '.')
    return ' '.join(parts)

def grounded_report_answer(image_id, groups, row):
    parts = [birads_density_answer(row)]
    items = image_to_boxes.get(image_id, [])
    if not groups or not items:
        parts.append('No suspicious mammographic finding is present.')
        return ' '.join(parts)
    for it in items:
        b = it['box']
        parts.append(f'{GROUP_TO_PHRASE[it["group"]]} <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
    return ' '.join(parts)

def build_examples_for_image(image_id, grounded=False):
    png_path = PNG / f'{image_id}.png'
    if not png_path.exists(): return []
    row = get_breast_row(image_id)
    if row is None: return []

    groups = image_to_groups.get(image_id, [])
    examples = []

    # Task 1: finding classification
    examples.append({
        'image': str(png_path), 'task': 'finding_classification',
        'messages': make_messages(png_path,
            'What suspicious mammographic findings are present?',
            finding_classification_answer(groups))
    })

    # Task 2: BI-RADS + density
    examples.append({
        'image': str(png_path), 'task': 'birads_density',
        'messages': make_messages(png_path,
            'What is the BI-RADS category and breast density?',
            birads_density_answer(row))
    })

    # Task 3: report
    if grounded:
        report = grounded_report_answer(image_id, groups, row)
        user = ('Generate a grounded mammography report. For each suspicious finding, '
                'append its location as <box>x_min y_min x_max y_max</box> in normalized 0-1000 coordinates.')
        task = 'grounded_report'
    else:
        report = plain_report_answer(groups, row)
        user = 'Generate a concise mammography finding report.'
        task = 'plain_report'
    examples.append({'image': str(png_path), 'task': task,
                     'messages': make_messages(png_path, user, report)})

    # Task 4: per-finding grounding (grounded only)
    if grounded:
        for it in image_to_boxes.get(image_id, []):
            g, b = it['group'], it['box']
            examples.append({
                'image': str(png_path), 'task': 'single_finding_grounding',
                'messages': make_messages(png_path,
                    f'Locate {g}. {GROUP_TO_KNOWLEDGE[g]}',
                    f'The {g} is located at <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
            })

    return examples

def write_split(ids, split_name, out_dir, grounded):
    out_path = out_dir / f'{split_name}.jsonl'
    n_img = n_ex = 0  # keeps track of num imgs and exs written to file
    with open(out_path, 'w') as f:
        for image_id in tqdm(sorted(ids), desc=f'{out_dir.name}/{split_name}'):
            exs = build_examples_for_image(image_id, grounded=grounded)
            if not exs: continue
            n_img += 1
            for ex in exs:
                f.write(json.dumps(ex) + '\n')
                n_ex += 1
    print(f'  {split_name}: images={n_img}, examples={n_ex}')

print('=== Plain grouped (Model A) ===')
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_split(ids, split, PLAIN_DIR, grounded=False)

print('\n=== Grounded grouped (Model B) ===')
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_split(ids, split, GND_DIR, grounded=True)

=== Plain grouped (Model A) ===


  train: images=14399, examples=43197


  val: images=1600, examples=4800


  test: images=4000, examples=12000

=== Grounded grouped (Model B) ===


  train: images=14399, examples=44922


  val: images=1600, examples=5022


  test: images=4000, examples=12482


In [ ]:
YOLO_OUT = YOLO_DIR / 'grouped'
YOLO_GROUP_TO_ID = {c: i for i, c in enumerate(GROUPED_CLASSES)}

def write_yolo_split(ids, split_name):
    img_dir = YOLO_OUT / split_name / 'images'
    lbl_dir = YOLO_OUT / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    n_img = n_box = 0
    for image_id in tqdm(sorted(ids), desc=f'YOLO {split_name}'):
        src = PNG / f'{image_id}.png'
        if not src.exists(): continue

        size = get_png_size(image_id)
        if size is None: continue
        W, H = size

        dst_img = img_dir / f'{image_id}.png'
        if not dst_img.exists():
            os.symlink(src, dst_img)   # symlink instead of copy (saves disk)

        lines = []
        for _, r in finding_df[finding_df['image_id'] == image_id].iterrows():
            if pd.isna(r.get('xmin', np.nan)): continue
            groups = map_to_groups(r['finding_categories'])
            if not groups: continue

            orig_w, orig_h = float(r['width']), float(r['height'])
            sx, sy = W / orig_w, H / orig_h
            x1 = max(0, min(W, float(r['xmin']) * sx))
            x2 = max(0, min(W, float(r['xmax']) * sx))
            y1 = max(0, min(H, float(r['ymin']) * sy))
            y2 = max(0, min(H, float(r['ymax']) * sy))
            bw, bh = x2 - x1, y2 - y1
            if bw <= 1 or bh <= 1: continue

            xc = ((x1 + x2) / 2) / W
            yc = ((y1 + y2) / 2) / H
            bw, bh = bw / W, bh / H

            for g in groups:
                lines.append(f'{YOLO_GROUP_TO_ID[g]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
                n_box += 1

        (lbl_dir / f'{image_id}.txt').write_text('\n'.join(lines))
        n_img += 1

    print(f'  {split_name}: images={n_img}, boxes={n_box}')

for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_yolo_split(ids, split)

yaml = (
    f'path: {YOLO_OUT}\n'
    f'train: train/images\n'
    f'val: val/images\n'
    f'test: test/images\n'
    f'nc: {len(GROUPED_CLASSES)}\n'
    f'names: {GROUPED_CLASSES}\n'
)
yaml_path = DATA_ROOT / 'mammo_grouped.yaml'
yaml_path.write_text(yaml)
print('YAML written:', yaml_path)

  train: images=14399, boxes=1725


  val: images=1600, boxes=222


  test: images=4000, boxes=482
YAML written: $VINDR_ROOT/mammo_grouped.yaml


In [ ]:
# Visually inspect 5 training labels to confirm boxes align with images
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

sample_files = list((YOLO_DIR / 'grouped' / 'train' / 'labels').glob('*.txt'))[:5]
for lbl_file in sample_files:
    image_id = lbl_file.stem
    img_path = YOLO_DIR / 'grouped' / 'train' / 'images' / f'{image_id}.png'
    img = PILImage.open(img_path)
    W, H = img.size
    
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img, cmap='gray')
    
    for line in lbl_file.read_text().strip().split('\n'):
        if not line: continue
        cls_id, xc, yc, bw, bh = line.split()
        cls_id, xc, yc, bw, bh = int(cls_id), float(xc), float(yc), float(bw), float(bh)
        x1 = (xc - bw/2) * W
        y1 = (yc - bh/2) * H
        ax.add_patch(Rectangle((x1, y1), bw*W, bh*H, linewidth=2, 
                               edgecolor='red', facecolor='none'))
        ax.text(x1, y1-5, GROUPED_CLASSES[cls_id], color='red', fontsize=10,
                bbox=dict(facecolor='black', alpha=0.7))
    ax.set_title(f'{image_id}')
    ax.axis('off')
    plt.show()

In [8]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Get only label files that are NON-EMPTY (have at least one box)
lbl_dir = YOLO_DIR / 'grouped' / 'train' / 'labels'
img_dir = YOLO_DIR / 'grouped' / 'train' / 'images'

nonempty_labels = [
    f for f in lbl_dir.glob('*.txt')
    if f.read_text().strip() != ''
]
print(f'Found {len(nonempty_labels)} training images with at least one annotated box')

# Take the first 5 of those
for lbl_file in nonempty_labels[:5]:
    image_id = lbl_file.stem
    img_path = img_dir / f'{image_id}.png'
    img = PILImage.open(img_path)
    W, H = img.size

    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img, cmap='gray')

    lines = lbl_file.read_text().strip().split('\n')
    for line in lines:
        cls_id, xc, yc, bw, bh = line.split()
        cls_id = int(cls_id)
        xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
        x1 = (xc - bw/2) * W
        y1 = (yc - bh/2) * H
        ax.add_patch(Rectangle((x1, y1), bw*W, bh*H, linewidth=2,
                               edgecolor='red', facecolor='none'))
        ax.text(x1, y1-5, GROUPED_CLASSES[cls_id], color='red', fontsize=10,
                bbox=dict(facecolor='black', alpha=0.7))
    ax.set_title(f'{image_id}  |  {len(lines)} finding(s)')
    ax.axis('off')
    plt.show()

Found 1269 training images with at least one annotated box


## 7. Train Qwen2.5-VL-7B — Model A (plain grouped)

In [ ]:
# import sys
# !{sys.executable} -m pip install -U transformers accelerate peft trl bitsandbytes qwen-vl-utils[decord]

In [ ]:
import torch, json, gc
from pathlib import Path
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

def load_jsonl(path):
    return Dataset.from_list([json.loads(l) for l in open(path)])

# ---- Change for Model A vs B ----
TIER =  'grounded_grouped'
# ---------------------------------------

DATA_DIR = LORA_DATA / TIER
OUT_DIR  = LORA_RUNS / f'qwen25vl7b_{TIER}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_ds = load_jsonl(DATA_DIR / 'train.jsonl')
val_ds   = load_jsonl(DATA_DIR / 'val.jsonl')

print('Model:', MODEL_ID, '| Tier:', TIER)
print('Train:', len(train_ds), 'Val:', len(val_ds))
print('Sample assistant text:', train_ds[0]['messages'][-1]['content'][0]['text'][:200])

<site-packages>/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

2026-07-15 16:53:19.028601: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: //public/apps/cuda/12.1/lib64:/lib/python3.9/site-packages/nvidia/cudnn/lib:/lib/:/opt/scyld/slurm/lib64
2026-07-15 16:53:19.028633: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False
print('Loaded.', 'Params:', sum(p.numel() for p in model.parameters()) / 1e9, 'B')

In [ ]:
def clean_messages(messages):
    """Remove None fields that HF Datasets adds during schema unification."""
    out = []
    for m in messages:
        new_content = []
        for c in m['content']:
            cleaned = {k: v for k, v in c.items() if v is not None}
            new_content.append(cleaned)
        out.append({'role': m['role'], 'content': new_content})
    return out

def collate_qwen(examples):
    messages = [clean_messages(ex['messages']) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
    image_inputs, video_inputs = process_vision_info(messages)
    batch = processor(text=texts, images=image_inputs, videos=video_inputs,
                      padding=True, return_tensors='pt')
    labels = batch['input_ids'].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    if image_token_id is not None:
        labels[labels == image_token_id] = -100
    batch['labels'] = labels
    return batch

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

cfg = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to='none',
    dataset_kwargs={'skip_prepare_dataset': True},
)

trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=train_ds, eval_dataset=val_ds,
    peft_config=lora,
    data_collator=collate_qwen,
    processing_class=processor,
)
trainer.train()
trainer.save_model(str(OUT_DIR))
print('Saved to', OUT_DIR)

## 9. Inference

In [3]:
from peft import PeftModel

# ---- choose which model ----
EVAL_TIER = 'grounded_grouped'
# ----------------------------

ADAPTER_DIR = LORA_RUNS / f'qwen25vl7b_{EVAL_TIER}'
EVAL_DATA   = LORA_DATA / EVAL_TIER / 'test.jsonl'
PRED_OUT    = LORA_RUNS / f'preds_{EVAL_TIER}.jsonl'

processor = AutoProcessor.from_pretrained(MODEL_ID)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('Loaded adapter:', ADAPTER_DIR)


Loaded adapter: $VINDR_ROOT/lora_runs/qwen25vl7b_model_c


In [ ]:
@torch.no_grad()
def generate_one(messages, max_new_tokens=256):
    # strip assistant turn for inference
    inf_messages = [m for m in messages if m['role'] != 'assistant']
    text = processor.apply_chat_template(inf_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(inf_messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = gen[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

# generate predictions on test set
test_examples = [json.loads(l) for l in open(EVAL_DATA)]
print(f'{len(test_examples)} test examples')

with open(PRED_OUT, 'w') as f:
    for ex in tqdm(test_examples, desc='Generating'):
        pred = generate_one(ex['messages'])
        gt = ex['messages'][-1]['content'][0]['text']
        out = {
            'image': ex['image'],
            'task': ex['task'],
            'gt': gt,
            'pred': pred,
        }
        f.write(json.dumps(out) + '\n')

print('Saved predictions to', PRED_OUT)

12482 test examples


Saved predictions to $VINDR_ROOT/lora_runs/preds_model_c.jsonl


## 10. Evaluation

In [5]:
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, cohen_kappa_score

preds = [json.loads(l) for l in open(PRED_OUT)]
by_task = defaultdict(list)
for p in preds: by_task[p['task']].append(p)
for t, items in by_task.items():
    print(f'{t:30s} {len(items)}')

# ---- helpers ----
GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

def extract_findings(text):
    t = text.lower()
    out = set()
    for g, kws in GROUP_KEYWORDS.items():
        if any(k in t for k in kws):
            out.add(g)
    return out

def to_binary_vec(groups):
    return [1 if g in groups else 0 for g in GROUPED_CLASSES]

BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])', re.I)
DENSITY_RE = re.compile(r'density\s*([abcd])', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')

def extract_birads(text):
    m = BIRADS_RE.search(text)
    return m.group(1) if m else None

def extract_density(text):
    m = DENSITY_RE.search(text)
    return m.group(1).upper() if m else None

def extract_boxes(text):
    return [list(map(int, m)) for m in BOX_RE.findall(text)]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

finding_classification         4000
birads_density                 4000
grounded_report                4000
single_finding_grounding       482


In [8]:
# Finding classification metrics
if 'finding_classification' in by_task:
    items = by_task['finding_classification']
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'\nFinding classification: macro F1 = {macro_f1:.3f}, micro F1 = {micro_f1:.3f}')
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(len(GROUPED_CLASSES))))
    for i, g in enumerate(GROUPED_CLASSES):
        print(f'  {g:30s} P={p[i]:.3f}  R={r[i]:.3f}  F1={f[i]:.3f}')

# BI-RADS + density
if 'birads_density' in by_task:
    items = by_task['birads_density']
    bg, bp, dg, dp = [], [], [], []
    for it in items:
        b_g, b_p = extract_birads(it['gt']), extract_birads(it['pred'])
        d_g, d_p = extract_density(it['gt']), extract_density(it['pred'])
        if b_g and b_p: bg.append(int(b_g)); bp.append(int(b_p))
        if d_g and d_p: dg.append(d_g); dp.append(d_p)
    print(f'\nBI-RADS:  acc={accuracy_score(bg, bp):.3f}, weighted kappa={cohen_kappa_score(bg, bp, weights="quadratic"):.3f}  ({len(bg)} valid)')
    print(f'Density:  acc={accuracy_score(dg, dp):.3f}, weighted kappa={cohen_kappa_score(dg, dp, weights="quadratic"):.3f}  ({len(dg)} valid)')

# Grounding IoU (Model B / grounded tier only)
if 'single_finding_grounding' in by_task:
    items = by_task['single_finding_grounding']
    ious = []
    for it in items:
        gt_b = extract_boxes(it['gt'])
        pr_b = extract_boxes(it['pred'])
        if not gt_b or not pr_b:
            ious.append(0.0); continue
        ious.append(iou(gt_b[0], pr_b[0]))
    ious = np.array(ious)
    print(f'\nGrounding: mean IoU = {ious.mean():.3f}, IoU@0.5 = {(ious >= 0.5).mean():.3f}  ({len(ious)} cases)')

# Report extraction (findings from generated report)
for task_name in ['plain_report', 'grounded_report']:
    if task_name not in by_task: continue
    items = by_task[task_name]
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    print(f'\n{task_name} (finding extraction from report): macro F1 = {f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}')

In [7]:
import re

GROUPED_CLASSES = [
    'Mass', 'Suspicious Calcification', 'Asymmetry',
    'Architectural Distortion', 'Suspicious Lymph Node', 'Other Suspicious Signs'
]

GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])', re.I)
DENSITY_RE = re.compile(r'density\s*([abcd])', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')

def extract_findings(text):
    t = text.lower()
    return {g for g, kws in GROUP_KEYWORDS.items() if any(k in t for k in kws)}

def to_binary_vec(groups):
    return [1 if g in groups else 0 for g in GROUPED_CLASSES]

def extract_birads(text):
    m = BIRADS_RE.search(text)
    return m.group(1) if m else None

def extract_density(text):
    m = DENSITY_RE.search(text)
    return m.group(1).upper() if m else None

def extract_boxes(text):
    return [list(map(int, m)) for m in BOX_RE.findall(text)]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

print('Eval helpers loaded.')

Eval helpers loaded.


# Module B Image testing

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# colors per finding class
GROUP_COLORS = {
    'Mass':                     'red',
    'Suspicious Calcification': 'yellow',
    'Asymmetry':                'cyan',
    'Architectural Distortion': 'magenta',
    'Suspicious Lymph Node':    'orange',
    'Other Suspicious Signs':   'lime',
}

def parse_box_with_label(pred_text):
    """Extract (group_label, [x1,y1,x2,y2]) pairs from predicted text.
    Looks for finding mentions followed by <box>...</box>."""
    results = []
    # split into sentences and look for group + box
    sentences = re.split(r'\.\s*', pred_text)
    for sent in sentences:
        boxes = BOX_RE.findall(sent)
        if not boxes:
            continue
        # find which group is mentioned
        sent_lower = sent.lower()
        matched_group = None
        for g, kws in GROUP_KEYWORDS.items():
            if any(k in sent_lower for k in kws):
                matched_group = g
                break
        for box in boxes:
            results.append((matched_group or 'Unknown', [int(x) for x in box]))
    return results

def denorm_box(box1000, png_w, png_h):
    """Convert 0-1000 normalized box to pixel coords."""
    return [
        box1000[0] / 1000 * png_w,
        box1000[1] / 1000 * png_h,
        box1000[2] / 1000 * png_w,
        box1000[3] / 1000 * png_h,
    ]

def visualize_predictions(image_id, pred_text, show_gt=True, figsize=(10, 12)):
    """Side-by-side: GT boxes (left) vs predicted boxes (right)."""
    img_path = PNG / f'{image_id}.png'
    img = Image.open(img_path)
    W, H = img.size

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # left: GT boxes
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'Ground Truth\n{image_id}')
    if show_gt:
        gt_items = image_to_boxes.get(image_id, [])
        for item in gt_items:
            g, b = item['group'], item['box']
            x1, y1, x2, y2 = denorm_box(b, W, H)
            color = GROUP_COLORS.get(g, 'white')
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                             edgecolor=color, facecolor='none')
            axes[0].add_patch(rect)
            axes[0].text(x1, y1-10, g, color=color, fontsize=9,
                         bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[0].axis('off')

    # right: predicted boxes
    axes[1].imshow(img, cmap='gray')
    axes[1].set_title('Model B Prediction')
    pred_items = parse_box_with_label(pred_text)
    for g, b in pred_items:
        x1, y1, x2, y2 = denorm_box(b, W, H)
        color = GROUP_COLORS.get(g, 'white')
        rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                         edgecolor=color, facecolor='none', linestyle='--')
        axes[1].add_patch(rect)
        axes[1].text(x1, y1-10, g, color=color, fontsize=9,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# ex usage - pick a test image that has findings
preds_grounded = [json.loads(l) for l in open(LORA_RUNS / 'preds_grounded_grouped.jsonl')]

# filter to grounded_report task only (has full box output)
report_preds = [p for p in preds_grounded if p['task'] == 'grounded_report']

# show 5 examples that have GT boxes
shown = 0
for p in report_preds:
    image_id = Path(p['image']).stem
    if image_id in image_to_boxes and shown < 5:
        print(f'\nGT report:   {p["gt"][:200]}')
        print(f'Pred report: {p["pred"][:200]}')
        visualize_predictions(image_id, p['pred'])
        shown += 1


GT report:   The BI-RADS category is BI-RADS 4. The breast density is DENSITY D. Suspicious calcifications are present <box>19 319 59 347</box>.
Pred report: The BI-RADS category is BI-RADS 2. The breast density is DENSITY C. No suspicious mammographic finding is present.



GT report:   The BI-RADS category is BI-RADS 3. The breast density is DENSITY B. A suspicious mass is present <box>21 572 73 620</box>.
Pred report: The BI-RADS category is BI-RADS 2. The breast density is DENSITY C. No suspicious mammographic finding is present.



GT report:   The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. A suspicious mass is present <box>827 399 903 461</box>.
Pred report: The BI-RADS category is BI-RADS 3. The breast density is DENSITY C. A suspicious mass is present <box>829 405 896 450</box>.



GT report:   The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. An asymmetry is present <box>15 481 145 615</box>.
Pred report: The BI-RADS category is BI-RADS 1. The breast density is DENSITY C. No suspicious mammographic finding is present.



GT report:   The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. Suspicious calcifications are present <box>104 312 254 492</box>.
Pred report: The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. Suspicious calcifications are present <box>108 325 267 451</box>.


# Report Generation

In [ ]:
# === quick recovery: redefine constants needed for inference/eval/viz ===

SYSTEM_MAMMO = (
    'You are an expert radiology assistant for mammography. '
    'Answer carefully using only the image information.'
)

PROMPT_REPORT = (
    'Generate a grounded mammography report. For each suspicious finding, '
    'append its location as <box>x_min y_min x_max y_max</box> in normalized 0-1000 coordinates.'
)

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

GROUPED_CLASSES = [
    'Mass', 'Suspicious Calcification', 'Asymmetry',
    'Architectural Distortion', 'Suspicious Lymph Node', 'Other Suspicious Signs',
]

GROUP_COLORS = {
    'Mass': 'red', 'Suspicious Calcification': 'yellow',
    'Asymmetry': 'cyan', 'Architectural Distortion': 'magenta',
    'Suspicious Lymph Node': 'orange', 'Other Suspicious Signs': 'lime',
}

GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

import re
BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])', re.I)
DENSITY_RE = re.compile(r'density\s*([abcd])', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')

def clean_messages(messages):
    out = []
    for m in messages:
        new_content = [{k: v for k, v in c.items() if v is not None} for c in m['content']]
        out.append({'role': m['role'], 'content': new_content})
    return out

print('Constants restored.')

Constants restored.


## NARRATIVE FORMAT FOR WHAT WE HAVE

In [ ]:
# ============================================================
# Mammography report generation — narrative format
# ============================================================

MAMMO_REPORT_TEMPLATE = """EXAM: Digital screening mammography, bilateral
TECHNIQUE: Standard bilateral digital mammography with craniocaudal (CC) and mediolateral oblique (MLO) views.

FINDINGS:
Breast composition: {density_text}

{findings_section}

IMPRESSION:
{birads_text}
{recommendation}
"""

# BI-RADS → clinical interpretation and recommendation
BIRADS_INFO = {
    '1': {
        'text': 'BI-RADS Category 1: Negative.',
        'rec':  'Routine annual screening mammography is recommended.',
    },
    '2': {
        'text': 'BI-RADS Category 2: Benign finding.',
        'rec':  'Routine annual screening mammography is recommended.',
    },
    '3': {
        'text': 'BI-RADS Category 3: Probably benign finding.',
        'rec':  'Short-interval follow-up mammography in 6 months is recommended.',
    },
    '4': {
        'text': 'BI-RADS Category 4: Suspicious abnormality.',
        'rec':  'Tissue diagnosis (biopsy) should be considered.',
    },
    '5': {
        'text': 'BI-RADS Category 5: Highly suggestive of malignancy.',
        'rec':  'Appropriate action (biopsy and/or treatment) should be taken.',
    },
}

DENSITY_INFO = {
    'A': 'The breasts are almost entirely fatty (BI-RADS density A).',
    'B': 'The breasts have scattered areas of fibroglandular density (BI-RADS density B).',
    'C': 'The breasts are heterogeneously dense, which may obscure small masses (BI-RADS density C).',
    'D': 'The breasts are extremely dense, which lowers the sensitivity of mammography (BI-RADS density D).',
}

FINDING_DESCRIPTIONS = {
    'Mass': 'A suspicious mass is identified',
    'Suspicious Calcification': 'Suspicious calcifications are noted',
    'Asymmetry': 'An asymmetry is present',
    'Architectural Distortion': 'Architectural distortion is identified',
    'Suspicious Lymph Node': 'A suspicious lymph node is noted',
    'Other Suspicious Signs': 'Additional suspicious mammographic signs are noted (such as skin thickening, nipple retraction, or skin retraction)',
}

def format_report(birads, density, findings_with_boxes, laterality_hint='right or left'):
    """Format a structured prediction into a narrative mammography report.
    
    findings_with_boxes: list of (group_name, [x1,y1,x2,y2]) tuples in 0-1000 coords
    """
    # Density section
    density_text = DENSITY_INFO.get(density, f'Breast density is {density}.')
    
    # Findings section
    if not findings_with_boxes:
        findings_section = (
            "No suspicious mass, suspicious calcification, asymmetry, architectural distortion, "
            "or other suspicious finding is identified."
        )
    else:
        finding_lines = []
        for g, box in findings_with_boxes:
            desc = FINDING_DESCRIPTIONS.get(g, f'A {g.lower()} is identified')
            # Convert 0-1000 box to a rough location description
            x_center = (box[0] + box[2]) / 2
            y_center = (box[1] + box[3]) / 2
            h_loc = 'medial' if x_center < 400 else ('lateral' if x_center > 600 else 'central')
            v_loc = 'upper' if y_center < 400 else ('lower' if y_center > 600 else 'middle')
            finding_lines.append(
                f"  - {desc} in the {v_loc} {h_loc} aspect of the breast "
                f"(normalized box coordinates: {box})."
            )
        findings_section = "The following finding(s) are noted:\n" + "\n".join(finding_lines)
    
    # impression
    birads_info = BIRADS_INFO.get(birads, {'text': f'BI-RADS Category {birads}.', 'rec': 'Clinical correlation recommended.'})
    
    return MAMMO_REPORT_TEMPLATE.format(
        density_text=density_text,
        findings_section=findings_section,
        birads_text=birads_info['text'],
        recommendation=birads_info['rec'],
    )

# test on a generated prediction
def generate_narrative_report(image_path):
    """End-to-end: image -> structured prediction -> narrative report."""
    msgs = clean_messages([
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_MAMMO}]},
        {'role': 'user',   'content': [{'type': 'image', 'image': str(image_path)},
                                       {'type': 'text', 'text': PROMPT_REPORT}]},
    ])
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(msgs)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    pred = processor.batch_decode(gen[:, inputs['input_ids'].shape[1]:],
                                  skip_special_tokens=True)[0].strip()
    
    # parse predictions
    birads  = extract_birads(pred) or 'unknown'
    density = extract_density(pred) or 'unknown'
    findings_with_boxes = parse_box_with_label(pred)
    
    narrative = format_report(birads, density, findings_with_boxes)
    return narrative, pred

# run on a test case
sample = next(p for p in report_preds if Path(p['image']).stem in image_to_boxes)
narrative, raw = generate_narrative_report(sample['image'])

print("=" * 70)
print("NARRATIVE REPORT")
print("=" * 70)
print(narrative)
print("\n" + "=" * 70)
print("RAW MODEL OUTPUT (for reference)")
print("=" * 70)
print(raw)

NARRATIVE REPORT
EXAM: Digital screening mammography, bilateral
TECHNIQUE: Standard bilateral digital mammography with craniocaudal (CC) and mediolateral oblique (MLO) views.

FINDINGS:
Breast composition: The breasts are heterogeneously dense, which may obscure small masses (BI-RADS density C).

No suspicious mass, suspicious calcification, asymmetry, architectural distortion, or other suspicious finding is identified.

IMPRESSION:
BI-RADS Category 2: Benign finding.
Routine annual screening mammography is recommended.


RAW MODEL OUTPUT (for reference)
The BI-RADS category is BI-RADS 2. The breast density is DENSITY C. No suspicious mammographic finding is present.


## Zero shot

In [ ]:
# ============================================================
# Zero-shot baseline — base Qwen2.5-VL, NO fine-tuning
# ============================================================
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

zs_processor = AutoProcessor.from_pretrained(MODEL_ID)
zs_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
zs_model.eval()

ZERO_SHOT_PROMPT = """You are a radiologist. Generate a structured mammography report for this image in the standard ACR BI-RADS format. Include:

1. EXAM and TECHNIQUE sections
2. FINDINGS section including:
   - Breast composition (density A, B, C, or D)
   - Description of any masses, calcifications, asymmetries, architectural distortion, or other suspicious findings
3. IMPRESSION section including:
   - Overall BI-RADS category (1-5)
   - Recommendation based on BI-RADS

Use standard radiology language. Be precise but concise."""

def zero_shot_report(image_path):
    msgs = [
        {'role': 'user', 'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': ZERO_SHOT_PROMPT},
        ]}
    ]
    text = zs_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(msgs)
    inputs = zs_processor(text=[text], images=image_inputs, videos=video_inputs,
                          padding=True, return_tensors='pt').to(zs_model.device)
    with torch.no_grad():
        gen = zs_model.generate(**inputs, max_new_tokens=500, do_sample=False)
    return zs_processor.batch_decode(gen[:, inputs['input_ids'].shape[1]:],
                                     skip_special_tokens=True)[0].strip()

# test on a sample image
sample_path = sample['image']
zs_output = zero_shot_report(sample_path)

print("=" * 70)
print("ZERO-SHOT (no fine-tuning) REPORT")
print("=" * 70)
print(zs_output)

ZERO-SHOT (no fine-tuning) REPORT
**ACR BI-RADS Mammography Report**

**EXAM:**
- **Patient ID:** [Patient ID]
- **Exam Date:** [Date]
- **Technique:** Craniocaudal (CC) view of the left breast.

**TECHNIQUE:**
- **Film Type:** Digital mammography.
- **Exposure Settings:** [Exposure settings].
- **Position:** Craniocaudal (CC) view of the left breast.

**FINDINGS:**
- **Breast Composition:** Density B (fibroglandular tissue with scattered areas of fat).
- **Masses:** No palpable masses noted on physical examination.
- **Calcifications:** No microcalcifications identified.
- **Asymmetries:** No significant asymmetry noted between the left and right breasts.
- **Architectural Distortion:** No evidence of architectural distortion.
- **Other Suspicious Findings:** A small, well-circumscribed, oval-shaped density is noted in the upper outer quadrant of the left breast. The density measures approximately 8 mm in diameter and has a smooth margin. No associated skin thickening or mass effect i

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage

sample_path = sample['image']
image_id = Path(sample_path).stem

# ---- 1. GROUND TRUTH ----
gt_row = breast_df[breast_df['image_id'] == image_id].iloc[0]
gt_birads_num = gt_row['breast_birads'].replace('BI-RADS ', '')   # '4'
gt_density_letter = gt_row['breast_density'].replace('DENSITY ', '')  # 'C'
gt_laterality = gt_row['laterality']
gt_view = gt_row['view_position']
gt_groups = image_to_groups.get(image_id, [])
gt_box_items = [(it['group'], it['box']) for it in image_to_boxes.get(image_id, [])]

gt_narrative = format_report(gt_birads_num, gt_density_letter, gt_box_items)

# ---- 2. ZERO-SHOT (base Qwen, no fine-tuning) ----
print("Running zero-shot...")
zs_narrative = zero_shot_report(sample_path)

# ---- 3. FINE-TUNED MODEL B (narrative format) ----
print("Running Model B narrative report...")
ft_narrative, ft_raw = generate_narrative_report(sample_path)

# ---- display image with GT boxes ----
img = PILImage.open(sample_path)
W, H = img.size
fig, ax = plt.subplots(1, 1, figsize=(7, 9))
ax.imshow(img, cmap='gray')
ax.set_title(f'{image_id}  |  {gt_laterality} {gt_view}', fontsize=11)
for g, b in gt_box_items:
    x1, y1, x2, y2 = denorm_box(b, W, H)
    color = GROUP_COLORS.get(g, 'white')
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=3, edgecolor=color, facecolor='none'))
    ax.text(x1, y1-10, f'GT: {g}', color=color, fontsize=10,
            bbox=dict(facecolor='black', alpha=0.7, pad=2))
ax.axis('off')
plt.tight_layout()
plt.show()

# ---- narrative reports ----
print("\n" + "=" * 75)
print("GROUND TRUTH (formatted from radiologist annotations)")
print("=" * 75)
print(gt_narrative)

print("\n" + "=" * 75)
print("ZERO-SHOT REPORT (Qwen2.5-VL base, NO fine-tuning)")
print("=" * 75)
print(zs_narrative)

print("\n" + "=" * 75)
print("MODEL B REPORT (Qwen2.5-VL + LoRA, grounded fine-tuning)")
print("=" * 75)
print(ft_narrative)

Running zero-shot...
Running Model B narrative report...



GROUND TRUTH (formatted from radiologist annotations)
EXAM: Digital screening mammography, bilateral
TECHNIQUE: Standard bilateral digital mammography with craniocaudal (CC) and mediolateral oblique (MLO) views.

FINDINGS:
Breast composition: The breasts are extremely dense, which lowers the sensitivity of mammography (BI-RADS density D).

The following finding(s) are noted:
  - Suspicious calcifications are noted in the upper medial aspect of the breast (normalized box coordinates: [19, 319, 59, 347]).

IMPRESSION:
BI-RADS Category 4: Suspicious abnormality.
Tissue diagnosis (biopsy) should be considered.


ZERO-SHOT REPORT (Qwen2.5-VL base, NO fine-tuning)
**ACR BI-RADS Mammography Report**

**EXAM:**
- **Patient ID:** [Patient ID]
- **Exam Date:** [Date]
- **Technique:** Craniocaudal (CC) view of the left breast.

**TECHNIQUE:**
- **Position:** Craniocaudal (CC) view.
- **Film Ty

... [long training log trimmed for the repo] ...

ing studies, such as ultrasound or MRI, to character

In [23]:
preds_grounded = [json.loads(l) for l in open(LORA_RUNS / 'preds_grounded_grouped.jsonl')]
report_preds = [p for p in preds_grounded if p['task'] == 'grounded_report']
print(f'{len(report_preds)} grounded report predictions loaded')

4000 grounded report predictions loaded


# YOLO EVAL (Trained in A)

In [ ]:
from ultralytics import YOLO

# load trained YOLO once (use best.pt from your training run)
YOLO_CKPT = RUNS / 'mammo_yolo11l_grouped' / 'weights' / 'best.pt'
yolo_infer = YOLO(str(YOLO_CKPT))
print(f'Loaded YOLO from {YOLO_CKPT}')

YOLO_ID_TO_GROUP = {i: c for i, c in enumerate(GROUPED_CLASSES)}

def yolo_predict_boxes(image_id, conf_threshold=0.25):
    """Run YOLO on a PNG, return list of (group_label, [x1,y1,x2,y2], confidence)."""
    img_path = PNG / f'{image_id}.png'
    results = yolo_infer.predict(str(img_path), conf=conf_threshold, verbose=False)
    out = []
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            xyxy = box.xyxy[0].tolist()  # pixel coords [x1,y1,x2,y2]
            out.append((YOLO_ID_TO_GROUP.get(cls_id, 'Unknown'), xyxy, conf))
    return out

def visualize_three_panel(image_id, pred_text, conf_threshold=0.25, figsize=(18, 10)):
    """GT | Model B | YOLO side-by-side."""
    img_path = PNG / f'{image_id}.png'
    img = Image.open(img_path)
    W, H = img.size

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # ---- Panel 1: GT ----
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'Ground Truth\n{image_id}', fontsize=11)
    for item in image_to_boxes.get(image_id, []):
        g, b = item['group'], item['box']
        x1, y1, x2, y2 = denorm_box(b, W, H)
        color = GROUP_COLORS.get(g, 'white')
        axes[0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor=color, facecolor='none'))
        axes[0].text(x1, y1-10, g, color=color, fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[0].axis('off')

    # ---- Panel 2: Model B ----
    axes[1].imshow(img, cmap='gray')
    axes[1].set_title('Model B (Grounded VLM)', fontsize=11)
    for g, b in parse_box_with_label(pred_text):
        x1, y1, x2, y2 = denorm_box(b, W, H)
        color = GROUP_COLORS.get(g, 'white')
        axes[1].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor=color, facecolor='none', linestyle='--'))
        axes[1].text(x1, y1-10, g, color=color, fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[1].axis('off')

    # ---- Panel 3: YOLO ----
    axes[2].imshow(img, cmap='gray')
    axes[2].set_title(f'YOLO (conf ≥ {conf_threshold})', fontsize=11)
    for g, xyxy, conf in yolo_predict_boxes(image_id, conf_threshold=conf_threshold):
        x1, y1, x2, y2 = xyxy
        color = GROUP_COLORS.get(g, 'white')
        axes[2].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor=color, facecolor='none', linestyle=':'))
        axes[2].text(x1, y1-10, f'{g} ({conf:.2f})', color=color, fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# run on some test cases that have GT findings
shown = 0
for p in report_preds:
    image_id = Path(p['image']).stem
    if image_id in image_to_boxes and shown < 5:
        print(f'\n=== {image_id} ===')
        print(f'GT:    {p["gt"][:180]}')
        print(f'Pred:  {p["pred"][:180]}')
        visualize_three_panel(image_id, p['pred'])
        shown += 1

Loaded YOLO from $VINDR_ROOT/runs/mammo_yolo11l_grouped/weights/best.pt

=== <image_id> ===
GT:    The BI-RADS category is BI-RADS 4. The breast density is DENSITY D. Suspicious calcifications are present <box>19 319 59 347</box>.
Pred:  The BI-RADS category is BI-RADS 2. The breast density is DENSITY C. No suspicious mammographic finding is present.



=== <image_id> ===
GT:    The BI-RADS category is BI-RADS 3. The breast density is DENSITY B. A suspicious mass is present <box>21 572 73 620</box>.
Pred:  The BI-RADS category is BI-RADS 2. The breast density is DENSITY C. No suspicious mammographic finding is present.



=== <image_id> ===
GT:    The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. A suspicious mass is present <box>827 399 903 461</box>.
Pred:  The BI-RADS category is BI-RADS 3. The breast density is DENSITY C. A suspicious mass is present <box>829 405 896 450</box>.



=== <image_id> ===
GT:    The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. An asymmetry is present <box>15 481 145 615</box>.
Pred:  The BI-RADS category is BI-RADS 1. The breast density is DENSITY C. No suspicious mammographic finding is present.



=== <image_id> ===
GT:    The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. Suspicious calcifications are present <box>104 312 254 492</box>.
Pred:  The BI-RADS category is BI-RADS 4. The breast density is DENSITY C. Suspicious calcifications are present <box>108 325 267 451</box>.


In [ ]:
import json, os
from collections import Counter

p = LORA_RUNS / 'preds_zero_shot.jsonl'
print(f'Size: {os.path.getsize(p) / 1024:.1f} KB')

lines = [json.loads(l) for l in open(p)]
print(f'Lines: {len(lines)}')

tasks = Counter(l['task'] for l in lines)
print(f'Tasks: {dict(tasks)}')

print(f'\nFirst example:')
print(f'  task: {lines[0]["task"]}')
print(f'  gt:   {lines[0]["gt"][:100]}')
print(f'  pred: {lines[0]["pred"][:200]}')

Size: 9768.5 KB
Lines: 12482
Tasks: {'finding_classification': 4000, 'birads_density': 4000, 'grounded_report': 4000, 'single_finding_grounding': 482}

First example:
  task: finding_classification
  gt:   No suspicious mammographic finding is present.
  pred: The provided image is a mammogram of the right breast in craniocaudal (CC) view. The image shows a portion of the breast tissue, but there are no obvious suspicious findings such as masses, calcificat


In [ ]:
import torch, json, gc
from pathlib import Path
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

try:
    del trainer, model, base, processor
except NameError: pass
gc.collect()
torch.cuda.empty_cache()

# load base model (no adapter)
MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
zs_processor = AutoProcessor.from_pretrained(MODEL_ID)
zs_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
zs_model.eval()
print('Base model loaded.')

def clean_messages(messages):
    out = []
    for m in messages:
        new_content = [{k: v for k, v in c.items() if v is not None} for c in m['content']]
        out.append({'role': m['role'], 'content': new_content})
    return out

@torch.no_grad()
def generate_zs(messages, max_new_tokens=100):
    inf_msgs = clean_messages([m for m in messages if m['role'] != 'assistant'])
    text = zs_processor.apply_chat_template(inf_msgs, tokenize=False, add_generation_prompt=True)
    img_in, vid_in = process_vision_info(inf_msgs)
    inputs = zs_processor(text=[text], images=img_in, videos=vid_in,
                          padding=True, return_tensors='pt').to(zs_model.device)
    gen = zs_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return zs_processor.batch_decode(gen[:, inputs['input_ids'].shape[1]:],
                                     skip_special_tokens=True)[0].strip()

EVAL_DATA = LORA_DATA / 'grounded_grouped' / 'test.jsonl'
PRED_OUT  = LORA_RUNS / 'preds_zero_shot.jsonl'

test_examples = [json.loads(l) for l in open(EVAL_DATA)]
print(f'Total: {len(test_examples)} examples (full test set)')

with open(PRED_OUT, 'w') as f:
    for ex in tqdm(test_examples, desc='Zero-shot'):
        try:
            pred = generate_zs(ex['messages'])
        except Exception as e:
            pred = f'[ERROR: {e}]'
        out = {
            'image': ex['image'],
            'task': ex['task'],
            'gt': ex['messages'][-1]['content'][0]['text'],
            'pred': pred,
        }
        f.write(json.dumps(out) + '\n')
        f.flush()

print(f'Saved to {PRED_OUT}')